# FER-YOLO-Mamba - RunPod Training (FER2013 / RTX 6000)

**Dataset:** upload your `split_FER2013_original` folder (or its zip) to `/workspace` on the pod.
Expected layout: `train/`, `val/`, `test/` with images + YOLO `.txt` labels.

Run every cell top to bottom.

In [ ]:
# CELL 1 - GPU CHECK
import torch, os, sys, subprocess

assert torch.cuda.is_available(), 'No GPU! Pick a GPU pod template on RunPod.'
p = torch.cuda.get_device_properties(0)
print('=' * 50)
print(f'  GPU     : {p.name}')
print(f'  VRAM    : {p.total_memory/1e9:.1f} GB')
print(f'  PyTorch : {torch.__version__}')
print(f'  CUDA    : {torch.version.cuda}')
print('=' * 50)

In [ ]:
# CELL 2 - CLONE CODE FROM GITHUB
REPO    = 'https://github.com/satyamsingh5512/FER-YOLO.git'
WORKDIR = '/workspace/FER-YOLO'

if os.path.exists(WORKDIR):
    print('Repo exists, pulling latest...')
    print(subprocess.run(['git', 'pull'], cwd=WORKDIR, capture_output=True, text=True).stdout)
else:
    r = subprocess.run(['git', 'clone', REPO, WORKDIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr)
    print('Cloned.')

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
print('Working dir:', os.getcwd())
print('Contents:', sorted(os.listdir('.')))

In [ ]:
# CELL 3 - INSTALL DEPENDENCIES (~10 min, compiles CUDA kernel)
for _p in ['/usr/local/cuda', '/usr/local/cuda-12', '/usr/local/cuda-12.9', '/usr/local/cuda-12.4']:
    if os.path.isdir(_p):
        os.environ['CUDA_HOME'] = _p
        os.environ['PATH'] = os.path.join(_p, 'bin') + ':' + os.environ.get('PATH', '')
        print('CUDA_HOME =', _p)
        break
os.environ['MAX_JOBS'] = '4'

def sh(cmd):
    print('$', ' '.join(cmd))
    return subprocess.run(cmd).returncode == 0

sh([sys.executable, '-m', 'pip', 'install', '-q', 'packaging', 'einops', 'timm>=0.6.13,<0.9.0'])

print('\nInstalling mamba-ssm (compiling, please wait ~10 min)...')
ok = (sh([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '--no-deps', 'mamba-ssm>=2.0.3'])
      or sh([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', 'mamba-ssm>=2.0.3']))
if not ok:
    raise RuntimeError('mamba-ssm install failed. Check the build log above.')

print('\nVerifying...')
import selective_scan_cuda, timm, einops
print('  selective_scan_cuda - OK')
print(f'  timm {timm.__version__} - OK')
print('All dependencies ready.')

In [ ]:
# CELL 4 - GET DATASET
# Downloads split_FER2013 from Google Drive automatically.
# (Make the Drive file shareable: Share -> Anyone with the link -> Viewer)
# If auto-download fails, upload the zip to /workspace manually and re-run.
import glob, zipfile

GDRIVE_ID = '1C9-WD-e4ojpZo9i6IxrG5lgiJY6RRYQk'
DATA_DIR  = '/workspace/dataset'
ZIP_PATH  = os.path.join(DATA_DIR, 'fer2013.zip')
os.makedirs(DATA_DIR, exist_ok=True)

# 1) download from Google Drive (skip if a dataset zip is already present)
existing_zip = glob.glob('/workspace/**/*fer*.zip', recursive=True) + \
               glob.glob('/workspace/**/*split*.zip', recursive=True)
if not os.path.exists(ZIP_PATH) and not existing_zip:
    sh([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    import gdown
    print('Downloading dataset from Google Drive...')
    gdown.download(id=GDRIVE_ID, output=ZIP_PATH, quiet=False)
elif existing_zip:
    ZIP_PATH = existing_zip[0]
    print('Using already-present zip:', ZIP_PATH)
else:
    print('Zip already downloaded:', ZIP_PATH)

# 2) unzip
if os.path.exists(ZIP_PATH):
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH) as f:
        f.extractall(DATA_DIR)
    print('Extracted to', DATA_DIR)

# 3) find dataset root (folder with train/ and (val/ or test/))
DATASET_ROOT = None
for base in ['/workspace', DATA_DIR]:
    if not os.path.isdir(base):
        continue
    for root, dirs, _ in os.walk(base):
        if root.replace(base, '').count(os.sep) > 5:
            dirs[:] = []
            continue
        if 'train' in dirs and ('val' in dirs or 'test' in dirs):
            DATASET_ROOT = root
            break
    if DATASET_ROOT:
        break

if not DATASET_ROOT:
    print('Could not auto-find dataset. /workspace contents:')
    for f in sorted(os.listdir('/workspace')): print('  ', f)
    raise RuntimeError(
        'Dataset not found. If the Drive download failed, make the file shareable '
        '(Anyone with the link) OR upload the zip to /workspace, then re-run.\n'
        "Or set DATASET_ROOT manually, e.g. '/workspace/dataset/split_FER2013_original'"
    )

print('DATASET_ROOT :', DATASET_ROOT)
print('Contents     :', sorted(os.listdir(DATASET_ROOT)))

In [ ]:
# CELL 5 - GENERATE ANNOTATION FILES (YOLO format -> repo format)
from convert_yolo_to_annotation import convert
convert(DATASET_ROOT, WORKDIR)

print()
for f in ['raf_train.txt', 'raf_val.txt']:
    lines = open(os.path.join(WORKDIR, f)).readlines()
    print(f'{f}: {len(lines)} samples')
    if lines:
        print('  sample ->', lines[0].strip()[:90])
    else:
        raise RuntimeError(f'{f} is empty! Check Cell 4 dataset structure.')

In [ ]:
# CELL 6 - PRE-FLIGHT CHECK
# Detect number of classes from the labels and pick the matching classes file
import collections
cls_ids = set()
for fn in ['raf_train.txt', 'raf_val.txt']:
    for line in open(fn):
        for box in line.strip().split()[1:]:
            cls_ids.add(int(box.split(',')[4]))
n_classes = (max(cls_ids) + 1) if cls_ids else 0
print(f'Class IDs present : {sorted(cls_ids)}  -> num_classes = {n_classes}')

# Use fer_classes.txt (7 emotions). If your labels have a different count,
# this writes a matching classes file automatically.
classes_path = 'model_data/fer_classes.txt'
existing = [l.strip() for l in open(classes_path)] if os.path.isfile(classes_path) else []
if len(existing) != n_classes:
    print(f'Adjusting classes file from {len(existing)} -> {n_classes} entries')
    with open(classes_path, 'w') as f:
        for i in range(n_classes):
            f.write(existing[i] if i < len(existing) else f'class{i}')
            f.write('\n')
os.environ['CLASSES_PATH'] = classes_path
print('CLASSES_PATH =', classes_path)

from nets.yolo import YoloBody
_ = YoloBody(n_classes, 's'); del _
print('YoloBody init - OK')
import selective_scan_cuda
print('selective_scan_cuda - OK')
print('\nReady to train.')

In [ ]:
# CELL 7 - TRAIN
# Batch sizes auto-tune to GPU VRAM. Override via env vars, e.g.:
#   os.environ['EPOCHS'] = '100'
#   os.environ['UNFREEZE_BATCH'] = '16'
env = dict(os.environ)   # passes CLASSES_PATH set in Cell 6
proc = subprocess.Popen(
    [sys.executable, '-u', 'train_kaggle.py'],
    cwd=WORKDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Training exited with code {proc.returncode}. Scroll up for the error.')
print('\nTraining finished.')

In [ ]:
# CELL 8 - RESULTS
import glob
ckpts = sorted(glob.glob('logs/**/*.pth', recursive=True))
print(f'Checkpoints ({len(ckpts)}):')
for c in ckpts:
    tag = ' <- BEST' if 'best' in c else (' <- LAST' if 'last' in c else '')
    print(f'  {c}  ({os.path.getsize(c)/1e6:.1f} MB){tag}')
print('\nDownload the logs/ folder from the RunPod file browser.')